# Entrenamiento YOLOv8s — Detector de Rayaduras en Baldosas
Ejecuta las celdas en orden. Tiempo estimado: **1-2 horas** en T4 gratuita.

In [ ]:
# CELDA 1 — Verificar GPU
import torch
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('AVISO: Sin GPU. Ve a Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# CELDA 2 — Instalar dependencias
!pip install ultralytics -q
print('Ultralytics OK')

In [ ]:
# CELDA 3 — Montar Google Drive (guarda checkpoints ahi para no perderlos)
from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/gdrive')
DRIVE_RUNS = Path('/gdrive/MyDrive/scratch_yolo_runs')
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
print(f'Checkpoints se guardaran en Google Drive: {DRIVE_RUNS}')

In [ ]:
# CELDA 4 — Subir scratch_data.zip directamente desde tu PC
# Se abrira un boton 'Choose Files' — selecciona scratch_data.zip
# (no hace falta Google Drive)
from google.colab import files
import os

ZIP_DEST = '/content/scratch_data.zip'

if os.path.exists(ZIP_DEST):
    print('ZIP ya existe en /content/, saltando subida.')
else:
    print('Selecciona scratch_data.zip en el dialogo...')
    uploaded = files.upload()          # abre el selector de archivos
    fname = list(uploaded.keys())[0]
    os.rename(fname, ZIP_DEST)
    print(f'Subido: {fname} -> {ZIP_DEST}')

size_mb = os.path.getsize(ZIP_DEST) / 1e6
print(f'ZIP listo en {ZIP_DEST} ({size_mb:.0f} MB)')

In [ ]:
# CELDA 5 — Extraer dataset
import zipfile
from pathlib import Path

DATA_ROOT = Path('/content/scratch_detector/data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print('Extrayendo ZIP...')
with zipfile.ZipFile('/content/scratch_data.zip') as z:
    z.extractall('/content/scratch_detector/')

print('Dataset extraido en', DATA_ROOT)
for split in ['train', 'val']:
    imgs = list((DATA_ROOT / 'images' / split).glob('*.jpg'))
    lbls = list((DATA_ROOT / 'labels' / split).glob('*.txt'))
    print(f'  {split}: {len(imgs)} imagenes, {len(lbls)} labels')

In [ ]:
# CELDA 6 — Fijar rutas en dataset.yaml
import yaml
from pathlib import Path

YAML_PATH = Path('/content/scratch_detector/data/dataset.yaml')
with open(YAML_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = '/content/scratch_detector/data'
with open(YAML_PATH, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)

print('dataset.yaml OK:')
print(f"  path:  {cfg['path']}")
print(f"  nc:    {cfg['nc']}")
print(f"  names: {cfg['names']}")

In [ ]:
# CELDA 7 — ENTRENAMIENTO (segmentacion YOLOv8s-seg)
# Si tienes last.pt de una sesion anterior, salta a la CELDA 7b (REANUDAR)
from ultralytics import YOLO
from pathlib import Path
import shutil, threading, time

YAML_PATH  = Path('/content/scratch_detector/data/dataset.yaml')
DRIVE_RUNS = Path('/gdrive/MyDrive/scratch_yolo_runs')

# Hilo que copia last.pt a Drive cada 10 minutos (por si se corta la sesion)
def _auto_backup():
    last = Path('/content/runs/scratch_yolo/weights/last.pt')
    while True:
        time.sleep(600)
        if last.exists():
            shutil.copy2(last, DRIVE_RUNS / 'last.pt')
            print('[BACKUP] last.pt guardado en Drive')

t = threading.Thread(target=_auto_backup, daemon=True)
t.start()

model = YOLO('yolov8s-seg.pt')

results = model.train(
    data          = str(YAML_PATH),
    epochs        = 150,
    imgsz         = 640,
    batch         = 16,
    patience      = 30,
    lr0           = 0.01,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15.0, translate=0.1, scale=0.5,
    flipud=0.3,   fliplr=0.5,
    mosaic=1.0,   mixup=0.1,
    box=7.5, cls=1.5, dfl=1.5,
    project       = '/content/runs',
    name          = 'scratch_yolo',
    save          = True,
    plots         = True,
    verbose       = True,
)

# Backup final
shutil.copy2('/content/runs/scratch_yolo/weights/best.pt', DRIVE_RUNS / 'best.pt')
shutil.copy2('/content/runs/scratch_yolo/weights/last.pt', DRIVE_RUNS / 'last.pt')
print('Entrenamiento completado — best.pt y last.pt guardados en Drive')

In [ ]:
# CELDA 7b — REANUDAR ENTRENAMIENTO desde last.pt
# Usa esta celda si la sesion anterior se corto.
# Sube primero last.pt usando el boton de archivos de Colab (icono carpeta > subir).
# O si lo tienes en Drive, se copia automaticamente.
from ultralytics import YOLO
from pathlib import Path
import shutil, os, threading, time

DRIVE_RUNS = Path('/gdrive/MyDrive/scratch_yolo_runs')
LAST_PT    = Path('/content/last.pt')

# Recuperar last.pt desde Drive si existe
if not LAST_PT.exists() and (DRIVE_RUNS / 'last.pt').exists():
    shutil.copy2(DRIVE_RUNS / 'last.pt', LAST_PT)
    print(f'last.pt recuperado desde Drive ({LAST_PT.stat().st_size/1e6:.0f} MB)')
elif LAST_PT.exists():
    print(f'last.pt encontrado localmente ({LAST_PT.stat().st_size/1e6:.0f} MB)')
else:
    raise FileNotFoundError('No se encontro last.pt. Subelo manualmente o ponlo en Drive.')

# Hilo de backup cada 10 min
def _auto_backup():
    last = Path('/content/runs/scratch_yolo/weights/last.pt')
    while True:
        time.sleep(600)
        if last.exists():
            shutil.copy2(last, DRIVE_RUNS / 'last.pt')
            print('[BACKUP] last.pt guardado en Drive')

t = threading.Thread(target=_auto_backup, daemon=True)
t.start()

# Reanudar — YOLO carga epoch, lr y pesos del checkpoint
model = YOLO(str(LAST_PT))
results = model.train(resume=True)

shutil.copy2('/content/runs/scratch_yolo/weights/best.pt', DRIVE_RUNS / 'best.pt')
shutil.copy2('/content/runs/scratch_yolo/weights/last.pt', DRIVE_RUNS / 'last.pt')
print('Entrenamiento completado — best.pt guardado en Drive')

In [ ]:
# CELDA 8 — Metricas finales
from ultralytics import YOLO
from pathlib import Path
import yaml

YAML_PATH = Path('/content/scratch_detector/data/dataset.yaml')
with open(YAML_PATH) as f:
    cfg = yaml.safe_load(f)

best = Path('/content/runs/scratch_yolo/weights/best.pt')
model_best = YOLO(str(best))
metrics = model_best.val(data=str(YAML_PATH), split='val')

names = cfg.get('names', {})
print(f'Box  mAP50:    {metrics.box.map50:.4f}')
print(f'Box  mAP50-95: {metrics.box.map:.4f}')
print(f'Mask mAP50:    {metrics.seg.map50:.4f}')
print(f'Mask mAP50-95: {metrics.seg.map:.4f}')
for i, ap in enumerate(metrics.seg.ap50):
    print(f'  Mask AP50[{i}] {names.get(i, i)}: {ap:.4f}')

In [ ]:
# CELDA 9 — Descargar best.pt al PC
from google.colab import files
from pathlib import Path

best_src = Path('/content/runs/scratch_yolo/weights/best.pt')
print(f'Descargando {best_src} ...')
files.download(str(best_src))
print()
print('Cuando se descargue, muevelo a:')
print('  scratch_detector/outputs/runs/scratch_yolo/weights/best.pt')